# TASK 2: - write a script that traders can use to price the contract

**important pointers from the task description:**
- any trade agreement is as valuable as the price you can sell minus the price at which you are able to buy. Any cost incurred as part of executing this agreement is also deducted from the overall value.
- ***A gas storage contract is worth the price difference between buying and selling, minus all operational costs.***
- from the given example, we can deduce the contract to look like :-
  
  $$Contract Value=(Sell Price−Buy Price)×Quantity−All Costs$$

## Objective
The aim of this task is to build a system that can estimate future natural gas prices and use those prices to evaluate the value of a storage contract. The idea is to simulate buying gas, storing it, and selling it later, while accounting for all associated costs.

---

## Data Overview
The dataset consists of monthly natural gas prices with two columns:

- Dates → month-end timestamps  
- FinalPrice → price of natural gas  

The data includes historical values from 2020 to 2024, and predicted values for 2024–2025.

---

## Approach

### 1. Forecasting Future Prices
The dataset shows both a trend and a seasonal pattern (prices tend to be higher in winter and lower in summer). To capture this, a SARIMA model was used.

The model was trained on historical monthly data and used to forecast prices for the next 12 months. This step is important because contract dates may fall beyond the available historical data.

---

### 2. Price Lookup
Since the data is monthly, prices are only available at month-end dates. To handle arbitrary input dates, each date is mapped to its nearest month-end and the corresponding price is used.

This keeps things consistent with the dataset without introducing unnecessary complexity.

---

### 3. Contract Pricing Logic
The contract is based on three main actions:

- Inject gas (buy)
- Store gas
- Withdraw gas (sell)

The value of the contract is calculated as:

Value = Revenue - Purchase Cost - Storage Cost

Revenue comes from selling gas, while costs include buying the gas and storing it over time.

---

### 4. Cash Flow Simulation
All injection and withdrawal events are processed in chronological order.

- On injection dates, gas is purchased and added to storage (cash outflow)
- On withdrawal dates, gas is sold and removed from storage (cash inflow)

At each step, storage costs are accumulated based on how long the gas is held and how much is stored.

---

### 5. Constraints
The system enforces a few practical constraints:

- Storage capacity cannot be exceeded  
- Cannot withdraw more gas than is stored  
- Injection and withdrawal happen at a fixed rate  

---

### 6. Storage Cost
Storage cost is calculated based on the duration for which gas is held and the amount stored. This ensures that holding gas longer increases total cost, which affects profitability.

---

## Observations
The data clearly shows seasonal behavior, with higher prices in winter and lower prices in summer. Because of this:

- Buying gas in summer and selling in winter tends to be profitable  
- Buying at high prices and selling later at lower prices leads to losses  

Negative contract values in some test cases simply indicate that the chosen strategy is not profitable.

---

## Comparison with Example Solution
The provided example solution assumes that prices for all dates are already known and focuses only on computing the contract value.

In this implementation, an additional step is included where future prices are estimated using a forecasting model. This makes the solution more realistic, since in practice future prices are not known in advance.

---

## Conclusion
The final system combines time-series forecasting with contract pricing logic. It first estimates future prices and then uses those prices to simulate buying, storing, and selling gas under realistic constraints.

This approach is more aligned with real-world scenarios where price estimation and decision-making go hand in hand.

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [7]:
df = pd.read_csv("final_gas_prices.csv")

df["Dates"] = pd.to_datetime(df["Dates"])
df.set_index("Dates", inplace=True)

ts = df["FinalPrice"].asfreq("ME")

print(ts.tail())

Dates
2025-05-31    12.231700
2025-06-30    12.276284
2025-07-31    12.260449
2025-08-31    12.305033
2025-09-30    12.349617
Freq: ME, Name: FinalPrice, dtype: float64


In [9]:
#implementing SARIMA to predict prices on the injection and withdrawal dates (Important Detail i added)
model = SARIMAX(ts,order=(1, 1, 1), seasonal_order=(1, 1, 1, 12), enforce_stationarity=False, enforce_invertibility=False)

results = model.fit()
forecast_steps = 12

future_index = pd.date_range(
    start=ts.index[-1] + pd.offsets.MonthEnd(1),
    periods=forecast_steps,
    freq="ME"
)

forecast = results.forecast(steps=forecast_steps)
forecast.index = future_index

full_series = pd.concat([ts, forecast])

print("\nFuture Forecast:")
print(forecast)


Future Forecast:
2025-10-31    13.018372
2025-11-30    13.272932
2025-12-31    13.613902
2026-01-31    13.725536
2026-02-28    13.640784
2026-03-31    13.820475
2026-04-30    13.060187
2026-05-31    12.710923
2026-06-30    12.784819
2026-07-31    12.830255
2026-08-31    12.798361
2026-09-30    12.978048
Freq: ME, Name: predicted_mean, dtype: float64


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [10]:
#price lookup function
def get_price(date):
    """returns pricing up to near date"""
    date = pd.to_datetime(date)

    # convert to month-end
    month_end = date + pd.offsets.MonthEnd(0)

    if month_end in full_series.index:
        return full_series.loc[month_end]
    else:
        raise ValueError(f"No price available for {month_end}")
        

In [11]:
#final contract pricing function
def price_contract(
    injection_dates,
    withdrawal_dates,
    quantity_per_action,
    max_storage,
    storage_cost_per_day
):
    events = []

    for d in injection_dates:
        events.append((pd.to_datetime(d), "inject"))

    for d in withdrawal_dates:
        events.append((pd.to_datetime(d), "withdraw"))

    events.sort()

    current_storage = 0
    total_value = 0
    total_cost = 0

    last_date = events[0][0]

    for date, action in events:

        # storage cost
        days = (date - last_date).days
        total_cost += days * storage_cost_per_day * current_storage

        price = get_price(date)

        if action == "inject":
            if current_storage + quantity_per_action > max_storage:
                raise ValueError("Storage capacity exceeded")

            current_storage += quantity_per_action
            total_value -= price * quantity_per_action

        elif action == "withdraw":
            if current_storage < quantity_per_action:
                raise ValueError("Not enough gas to withdraw")

            current_storage -= quantity_per_action
            total_value += price * quantity_per_action

        last_date = date

    return total_value - total_cost

In [14]:
#test and implementation
injection_dates = ["2025-01-15", "2025-02-10"]
withdrawal_dates = ["2025-03-20", "2025-04-15"]

contract_value = price_contract(
    injection_dates=injection_dates,
    withdrawal_dates=withdrawal_dates,
    quantity_per_action=100000,
    max_storage=500000,
    storage_cost_per_day=0.02
)

print("\nContract Value:", contract_value)


Contract Value: -336652.68619509204
